# Telecom Churn: Imbalance-Aware Models and Stacking with a Genetic-Programming Meta-Learner

Which telecom customers are about to leave? This notebook compares imbalance-aware classifiers, explains the strongest single model with SHAP, and tests whether stacking helps, including a stack whose meta-learner is a symbolic classifier evolved by genetic programming (`gplearn`).

All preprocessing lives inside scikit-learn pipelines, so encoders and scalers are fitted on training data only. Models are compared on a held-out test set with ROC AUC, recall, F1, cumulative gains and calibration, and the two best are compared with McNemar's test.

**Data:** IBM's [Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d/blob/master/data/Telco-Customer-Churn.csv) sample (Apache-2.0): 7,043 customers, 26.5% churn.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from gplearn.genetic import SymbolicClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.pipeline import make_pipeline as imb_pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn import metrics
from sklearn.calibration import CalibrationDisplay
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from statsmodels.stats.contingency_tables import mcnemar
from xgboost import XGBClassifier

SEED = 2022
sns.set_theme(style="whitegrid")

## Data

In [ ]:
URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
path = Path("data/Telco-Customer-Churn.csv")
if not path.exists():
    path.parent.mkdir(exist_ok=True)
    urlretrieve(URL, path)

ds = pd.read_csv(path)
ds["TotalCharges"] = pd.to_numeric(ds["TotalCharges"], errors="coerce")
print("blank TotalCharges (new customers, tenure 0):", ds["TotalCharges"].isna().sum())
ds["TotalCharges"] = ds["TotalCharges"].fillna(0)
ds["Churn"] = (ds["Churn"] == "Yes").astype(int)
ds = ds.drop(columns="customerID")
print(ds.shape)
ds["Churn"].value_counts(normalize=True)

## Exploration

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(data=ds, x="tenure", hue="Churn", multiple="stack", bins=36, ax=axes[0])
sns.histplot(data=ds, x="MonthlyCharges", hue="Churn", multiple="stack", bins=30, ax=axes[1])
(ds.groupby("Contract")["Churn"].mean().sort_values()
   .plot.barh(ax=axes[2], title="Churn rate by contract"))
plt.tight_layout()
plt.show()

In [ ]:
categorical = [c for c in ds.columns if ds[c].dtype == object]
rates = pd.concat({c: ds.groupby(c)["Churn"].mean() for c in categorical}).rename("churn rate")
rates.sort_values(ascending=False).head(12).to_frame()

Churn concentrates in new customers, month-to-month contracts, fibre-optic internet and electronic-check payment.

## Preprocessing and split

In [ ]:
y = ds.pop("Churn")
numeric = ["tenure", "MonthlyCharges", "TotalCharges"]
ordinal = ["Contract"]
nominal = [c for c in ds.columns if c not in numeric + ordinal]

def preprocess():
    return make_column_transformer(
        (StandardScaler(), numeric),
        (OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]]), ordinal),
        (OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), nominal),
    )

X_train, X_test, y_train, y_test = train_test_split(
    ds, y, test_size=0.2, stratify=y, random_state=SEED)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(X_train.shape, X_test.shape, f"negative/positive ratio {pos_weight:.2f}")

## Models

In [ ]:
def xgb():
    return XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.9,
                         colsample_bytree=0.9, scale_pos_weight=pos_weight, eval_metric="logloss",
                         random_state=SEED)

base_estimators = [
    ("rf", RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=SEED, n_jobs=-1)),
    ("xgb", xgb()),
]

models = {
    "Logistic regression": make_pipeline(preprocess(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Random forest": make_pipeline(preprocess(), RandomForestClassifier(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1)),
    "Balanced random forest": make_pipeline(preprocess(), BalancedRandomForestClassifier(
        n_estimators=300, max_depth=8, sampling_strategy="all", replacement=True, bootstrap=False,
        random_state=SEED, n_jobs=-1)),
    "Under-sampling + random forest": imb_pipeline(preprocess(), RandomUnderSampler(random_state=SEED),
        RandomForestClassifier(n_estimators=300, max_depth=8, random_state=SEED, n_jobs=-1)),
    "XGBoost": make_pipeline(preprocess(), xgb()),
    "Stacking (logistic meta-learner)": make_pipeline(preprocess(), StackingClassifier(
        base_estimators, final_estimator=LogisticRegression(class_weight="balanced"), cv=5)),
    "Stacking (genetic-programming meta-learner)": make_pipeline(preprocess(), StackingClassifier(
        base_estimators, cv=5, final_estimator=SymbolicClassifier(
            population_size=500, generations=40, tournament_size=10, parsimony_coefficient=0.001,
            class_weight="balanced", random_state=SEED))),
}

rows, probas = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    probas[name] = model.predict_proba(X_test)[:, 1]
    rows[name] = {
        "Accuracy": metrics.accuracy_score(y_test, pred),
        "Precision": metrics.precision_score(y_test, pred),
        "Recall": metrics.recall_score(y_test, pred),
        "F1": metrics.f1_score(y_test, pred),
        "ROC AUC": metrics.roc_auc_score(y_test, probas[name]),
        "Brier": metrics.brier_score_loss(y_test, probas[name]),
    }
results = pd.DataFrame(rows).T.sort_values("ROC AUC", ascending=False)
results.round(3)

The evolved meta-learner is a readable formula over the base models' churn probabilities (`X0` = random forest, `X1` = XGBoost):

In [ ]:
print(models["Stacking (genetic-programming meta-learner)"][-1].final_estimator_._program)

## Cumulative gains and calibration

In [ ]:
def gains(y_true, score):
    order = np.argsort(-score)
    caught = np.cumsum(np.asarray(y_true)[order]) / np.sum(y_true)
    return np.linspace(0, 1, len(caught)), caught

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for name, p in probas.items():
    axes[0].plot(*gains(y_test, p), label=name)
axes[0].plot([0, 1], [0, 1], "k--", label="Random")
axes[0].set(xlabel="Share of customers contacted", ylabel="Share of churners caught",
            title="Cumulative gains")
axes[0].legend(fontsize=8)
for name in ["Logistic regression", "XGBoost", "Stacking (genetic-programming meta-learner)"]:
    CalibrationDisplay.from_predictions(y_test, probas[name], n_bins=10, name=name, ax=axes[1])
axes[1].set_title("Calibration")
plt.tight_layout()
plt.show()

for name, p in probas.items():
    x, c = gains(y_test, p)
    print(f"{name:45s} churners caught in top 20%: {c[int(0.2 * len(c)) - 1]:.1%}")

## What drives churn: SHAP on XGBoost

In [ ]:
pipe = models["XGBoost"]
prep, booster = pipe[0], pipe[-1]
X_test_t = pd.DataFrame(prep.transform(X_test), columns=prep.get_feature_names_out())
X_test_t.columns = [c.split("__", 1)[1] for c in X_test_t.columns]
shap_values = shap.TreeExplainer(booster).shap_values(X_test_t)
shap.summary_plot(shap_values, X_test_t, max_display=15, show=False)
plt.tight_layout()
plt.show()

In [ ]:
pd.Series(np.abs(shap_values).mean(axis=0), index=X_test_t.columns).sort_values(ascending=False).head(8).round(3)

## Are the two best models really different? McNemar's test

In [ ]:
best_two = results.index[:2].tolist()
a = models[best_two[0]].predict(X_test) == y_test
b = models[best_two[1]].predict(X_test) == y_test
table = pd.crosstab(a, b).reindex(index=[True, False], columns=[True, False], fill_value=0)
print(best_two)
print(table)
test = mcnemar(table.to_numpy(), exact=False, correction=True)
print(f"McNemar chi2 = {test.statistic:.2f}, p = {test.pvalue:.3f}")

In [ ]:
# Sanity checks: every model ran and the best one is clearly better than chance.
assert len(results) == len(models)
assert results["ROC AUC"].max() > 0.8

## Conclusion

CONCLUSION_PLACEHOLDER